# 偏差与方差如何权衡？

**面试回答主线：**偏差是模型在平均意义上的系统误差，方差是模型随训练样本改变而波动的程度。不能只看训练集分数：训练和验证都差更像高偏差，训练好而验证差更像高方差。本实验用配送距离预测时长，比较常数、直线、高阶多项式和带 L2 的高阶模型。小数据只演示机制。

## 真实案例

12 条历史订单来自同一城市晴天午高峰，距离与配送时长存在平滑但非完全线性的关系。前 8 条作为历史训练，后 4 条是未来日期回放，模拟时间外验证而非随机切分。

In [1]:
import numpy as np  # 导入 NumPy 以手写回归与线性代数。
np.set_printoptions(precision=3, suppress=True)  # 设置紧凑的数值显示格式。
order_id = np.array(['A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'B01', 'B02', 'B03', 'B04'])  # 构造真实风格的订单编号。
distance = np.array([1.0, 1.8, 2.6, 3.4, 4.1, 4.9, 5.7, 6.5, 1.4, 3.0, 5.2, 7.0])  # 记录配送距离公里数。
minutes = np.array([18.0, 20.0, 24.0, 27.0, 31.0, 38.0, 45.0, 53.0, 19.0, 25.0, 41.0, 60.0])  # 记录实际送达分钟数。
train_index = np.arange(8)  # 将早日期的八条订单定义为训练集。
valid_index = np.arange(8, 12)  # 将未来日期的四条订单定义为验证集。
print('订单 | 距离km | 实际分钟 | 数据角色')  # 输出原始订单表头。
for index in range(len(order_id)):  # 逐行展示带业务语义的数据。
    role = '训练' if index in train_index else '未来回放'  # 标注当前订单属于训练还是未来验证。
    print(f'{order_id[index]} | {distance[index]:4.1f} | {minutes[index]:4.1f} | {role}')  # 输出一条订单记录。

订单 | 距离km | 实际分钟 | 数据角色
A01 |  1.0 | 18.0 | 训练
A02 |  1.8 | 20.0 | 训练
A03 |  2.6 | 24.0 | 训练
A04 |  3.4 | 27.0 | 训练
A05 |  4.1 | 31.0 | 训练
A06 |  4.9 | 38.0 | 训练
A07 |  5.7 | 45.0 | 训练
A08 |  6.5 | 53.0 | 训练
B01 |  1.4 | 19.0 | 未来回放
B02 |  3.0 | 25.0 | 未来回放
B03 |  5.2 | 41.0 | 未来回放
B04 |  7.0 | 60.0 | 未来回放


## Baseline / 基线

常数模型只预测历史平均配送时长，它不使用距离，因此代表过于简单的高偏差方案。

In [2]:
train_y = minutes[train_index]  # 提取训练订单的真实时长。
valid_y = minutes[valid_index]  # 提取未来回放订单的真实时长。
constant_prediction = np.full(len(valid_index), train_y.mean())  # 用训练集均值作为所有未来订单的预测。
constant_mse = float(np.mean((constant_prediction - valid_y) ** 2))  # 计算常数模型的验证均方误差。
print('常数基线预测:', np.round(constant_prediction, 1))  # 展示基线对每个未来订单的预测。
print(f'常数基线验证 MSE: {constant_mse:.2f}')  # 输出高偏差基线指标。

常数基线预测: [32. 32. 32. 32.]
常数基线验证 MSE: 270.75


In [3]:
def design_matrix(values, degree):  # 定义手写多项式特征矩阵函数。
    columns = [np.ones(len(values))]  # 先放入截距列。
    for power in range(1, degree + 1):  # 依次构造一阶到指定阶数的特征。
        columns.append(values ** power)  # 添加当前幂次的距离特征。
    return np.column_stack(columns)  # 将所有特征列拼成设计矩阵。
def fit_ridge(x_train, y_train, l2_strength):  # 定义带 L2 正则的正规方程求解函数。
    penalty = np.eye(x_train.shape[1]) * l2_strength  # 构造各权重的 L2 惩罚矩阵。
    penalty[0, 0] = 0.0  # 排除截距以避免整体均值被不必要压缩。
    return np.linalg.solve(x_train.T @ x_train + penalty, x_train.T @ y_train)  # 解线性方程得到回归系数。
def mse(prediction, target):  # 定义可复用的均方误差计算函数。
    return float(np.mean((prediction - target) ** 2))  # 返回平均平方残差。
print('设计矩阵示例，三阶前两行:', design_matrix(distance[:2], 3))  # 展示显式特征展开的中间量。

设计矩阵示例，三阶前两行: [[1.    1.    1.    1.   ]
 [1.    1.8   3.24  5.832]]


In [4]:
models = []  # 创建用于汇总不同容量模型的结果列表。
for degree, l2_strength, label in [(1, 0.0, '直线'), (5, 0.0, '五阶'), (5, 8.0, '五阶+L2')]:  # 比较低容量、高容量与正则化方案。
    x_train = design_matrix(distance[train_index], degree)  # 构造当前模型的训练设计矩阵。
    x_valid = design_matrix(distance[valid_index], degree)  # 构造当前模型的验证设计矩阵。
    weight = fit_ridge(x_train, train_y, l2_strength)  # 根据训练订单拟合模型参数。
    train_prediction = x_train @ weight  # 计算训练订单预测以观察拟合程度。
    valid_prediction = x_valid @ weight  # 计算未来订单预测以衡量泛化。
    models.append((label, degree, l2_strength, mse(train_prediction, train_y), mse(valid_prediction, valid_y), weight, valid_prediction))  # 保存可比较的指标与中间参数。
print('名称       | 训练MSE | 未来MSE | 系数L2范数')  # 输出模型比较表头。
for label, degree, l2_strength, train_error, valid_error, weight, prediction in models:  # 逐个输出容量比较结果。
    print(f'{label:10s} | {train_error:7.2f} | {valid_error:7.2f} | {np.linalg.norm(weight):9.2f}')  # 输出训练误差、验证误差和权重尺度。

名称       | 训练MSE | 未来MSE | 系数L2范数
直线         |    5.66 |   15.81 |     10.38
五阶         |    0.10 |    4.34 |     23.76
五阶+L2      |    0.25 |    0.67 |     18.09


## 结果解读

常数模型忽略距离，训练和未来误差都大，体现高偏差。五阶模型可以把训练点几乎穿过，但未来误差和权重范数可能上升，体现对样本扰动敏感。L2 不保证每次都胜出，却会抑制巨大高阶系数，是降低方差的机制。

In [5]:
linear_result = models[0]  # 读取直线模型的汇总结果。
complex_result = models[1]  # 读取未正则五阶模型的汇总结果。
ridge_result = models[2]  # 读取带正则五阶模型的汇总结果。
print('未来订单 | 真实分钟 | 直线预测 | 五阶预测 | 五阶+L2')  # 输出逐订单结果表头。
for local_index, global_index in enumerate(valid_index):  # 逐条比较未来订单预测。
    print(f'{order_id[global_index]} | {valid_y[local_index]:8.1f} | {linear_result[6][local_index]:8.1f} | {complex_result[6][local_index]:8.1f} | {ridge_result[6][local_index]:8.1f}')  # 输出同一订单的不同模型结果。
print('结论：应基于独立时间回放的误差和波动选容量，不以训练集最低损失决定。')  # 给出可复述的结果解释。

未来订单 | 真实分钟 | 直线预测 | 五阶预测 | 五阶+L2
B01 |     19.0 |     17.1 |     18.9 |     19.2
B02 |     25.0 |     27.2 |     25.3 |     24.8
B03 |     41.0 |     41.2 |     40.3 |     40.4
B04 |     60.0 |     52.6 |     55.9 |     58.5
结论：应基于独立时间回放的误差和波动选容量，不以训练集最低损失决定。


## 失败案例与修复

故意在仅 8 条训练订单上拟合九阶多项式：它会产生近乎零的训练误差，却对距离 7km 的未来订单给出不稳定外推。修复并非盲目加数据，而是先保留时间切分，再降低容量或加入正则，并持续监控跨窗口方差。

In [6]:
bad_x_train = design_matrix(distance[train_index], 7)  # 构造接近样本数的七阶特征以故意提升容量。
bad_x_valid = design_matrix(distance[valid_index], 7)  # 构造同样的未来验证特征。
bad_weight = fit_ridge(bad_x_train, train_y, 0.0)  # 在高容量且无正则条件下拟合模型。
bad_train_mse = mse(bad_x_train @ bad_weight, train_y)  # 计算近似记忆训练样本的误差。
bad_valid_mse = mse(bad_x_valid @ bad_weight, valid_y)  # 计算高方差模型的未来误差。
fixed_weight = fit_ridge(bad_x_train, train_y, 40.0)  # 给相同高阶特征加入较强 L2 正则。
fixed_valid_mse = mse(bad_x_valid @ fixed_weight, valid_y)  # 计算正则化后的未来误差。
print(f'失败：七阶无正则 训练MSE={bad_train_mse:.4f}，未来MSE={bad_valid_mse:.2f}')  # 输出过拟合的可复现证据。
print(f'修复：七阶加L2 后未来MSE={fixed_valid_mse:.2f}')  # 输出正则化的修复结果。
print('生产差距：真实配送还需天气、时段、骑手供给、预测区间、漂移监控和按城市回滚。')  # 说明教学实验尚未覆盖的生产能力。

失败：七阶无正则 训练MSE=0.0000，未来MSE=15.33
修复：七阶加L2 后未来MSE=0.90
生产差距：真实配送还需天气、时段、骑手供给、预测区间、漂移监控和按城市回滚。


In [7]:
assert len(order_id) >= 5  # 保护教学案例至少包含五条业务样本。
assert constant_mse > linear_result[4]  # 保护直线模型在该平滑任务上优于常数基线。
assert bad_train_mse < bad_valid_mse  # 保护高阶模型的训练—未来泛化间隙。
assert np.linalg.norm(ridge_result[5]) < np.linalg.norm(complex_result[5])  # 保护 L2 会缩小高阶模型权重这一机制。